1) Create a DataFrame for each file.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BikeTripGraph") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.2-s_2.12") \
    .getOrCreate()

In [ ]:
from google.colab import files

print("station_data.csv")
uploaded = files.upload()

print("trip_data.csv")
uploaded = files.upload()

station_data.csv


Saving station_data.csv to station_data.csv
trip_data.csv


Saving trip_data.csv to trip_data.csv


In [ ]:
stations_df = spark.read.csv("/content/station_data.csv", header=True, inferSchema=True)

trips_df = spark.read.csv("/content/trip_data.csv", header=True, inferSchema=True)

print("Stations DataFrame:")
stations_df.show(5)

print("Trips DataFrame:")
trips_df.show(5)

Stations DataFrame:
+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                name|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|San Jose|    8/6/2013|
|         5|    Adobe on Almaden|37.331415|  -121.8932|       19|San Jose|    8/5/2013|
|         6|    San Pedro Square|37.336721|-121.894074|       15|San Jose|    8/7/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 5 rows

Trips DataFrame:
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+---

2) In the vertex table (the DataFrame representing the file station_data.csv), rename the
column name to id.

In [ ]:
vertices_df = stations_df.withColumnRenamed("name", "id")
vertices_df.show(1)

Vertices DataFrame with 'id' column:
+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                  id|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 1 row



3) In the edge table (the DataFrame representing the file trip_data.csv), rename the
column Start Station to src and the column End Station to dst.

In [ ]:
edges_df = trips_df.withColumnRenamed("Start Station", "src").withColumnRenamed("End Station", "dst")
edges_df.show(1)

+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
|Trip ID|Duration|     Start Date|                 src|Start Terminal|       End Date|                 dst|End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
| 913460|     765|8/31/2015 23:26|Harry Bridges Pla...|            50|8/31/2015 23:39|San Francisco Cal...|          70|   288|     Subscriber|    2139|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
only showing top 1 row



4) Create a GraphFrame that represents our graph.

In [ ]:
from graphframes import GraphFrame

graph = GraphFrame(vertices_df, edges_df)

print("Graph Vertices:")
graph.vertices.show(5)

print("Graph Edges:")
graph.edges.show(5)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


Graph Vertices:
+----------+--------------------+---------+-----------+---------+--------+------------+
|station_id|                  id|      lat|       long|dockcount|landmark|installation|
+----------+--------------------+---------+-----------+---------+--------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|San Jose|    8/6/2013|
|         5|    Adobe on Almaden|37.331415|  -121.8932|       19|San Jose|    8/5/2013|
|         6|    San Pedro Square|37.336721|-121.894074|       15|San Jose|    8/7/2013|
+----------+--------------------+---------+-----------+---------+--------+------------+
only showing top 5 rows

Graph Edges:
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
|

5) Return the number of trips made between each source and destination, sorted in
descending order (based on the number of trips).

In [ ]:
trip_counts = graph.edges.groupBy("src", "dst").count().orderBy("count", ascending=False)

print("Number of trips between each source and destination =>")
trip_counts.show()

Number of trips between each source and destination =>
+--------------------+--------------------+-----+
|                 src|                 dst|count|
+--------------------+--------------------+-----+
|San Francisco Cal...|     Townsend at 7th|    4|
|       5th at Howard|San Francisco Cal...|    3|
|San Francisco Cal...|  Powell Street BART|    2|
|     2nd at Townsend|   Market at Sansome|    2|
|     Spear at Folsom|     2nd at Townsend|    2|
|   Market at Sansome|Broadway St at Ba...|    2|
|    Davis at Jackson|Embarcadero at Sa...|    2|
|San Francisco Cal...|   2nd at South Park|    2|
|   Steuart at Market|San Francisco Cal...|    2|
|       Market at 4th|San Francisco Cal...|    2|
|Embarcadero at Fo...|Embarcadero at Sa...|    2|
|Mountain View Cal...|Rengstorff Avenue...|    1|
|    Davis at Jackson|Temporary Transba...|    1|
|       5th at Howard|     Townsend at 7th|    1|
|   Market at Sansome|South Van Ness at...|    1|
|     Beale at Market|Temporary Transba...|  

6) Return the number of trips that start or end at the station "Townsend at 7th", sorted in
descending order (based on the number of trips).

In [ ]:
townsend_trips = graph.edges.filter("src = 'Townsend at 7th' OR dst = 'Townsend at 7th'") \
    .groupBy("src", "dst").count().orderBy("count", ascending=False)

print("Number of trips starting or ending at 'Townsend at 7th' =>")
townsend_trips.show()

Number of trips starting or ending at 'Townsend at 7th' =>
+--------------------+--------------------+-----+
|                 src|                 dst|count|
+--------------------+--------------------+-----+
|San Francisco Cal...|     Townsend at 7th|    4|
|       5th at Howard|     Townsend at 7th|    1|
|     Townsend at 7th|     Spear at Folsom|    1|
|     Townsend at 7th|Harry Bridges Pla...|    1|
|     Spear at Folsom|     Townsend at 7th|    1|
|       Howard at 2nd|     Townsend at 7th|    1|
+--------------------+--------------------+-----+



7) Return the vertices that have never been a destination for a trip starting from "Spear at
Folsom".

In [ ]:
spear_destinations = graph.edges.filter("src = 'Spear at Folsom'").select("dst").distinct()

non_destinations = graph.vertices.join(spear_destinations,
                                       graph.vertices.id == spear_destinations.dst,
                                       "left_anti") #"left anti-join" keeps only rows from graph.vertices that don’t match spear_destinations.



print("the vertices that have never been a destination for a trip starting from Spear atFolsom")
non_destinations.select("id").show()

the vertices that have never been a destination for a trip starting from Spear atFolsom
+--------------------+
|                  id|
+--------------------+
|San Jose Diridon ...|
|San Jose Civic Ce...|
|Santa Clara at Al...|
|    Adobe on Almaden|
|    San Pedro Square|
|Paseo de San Antonio|
| San Salvador at 1st|
|           Japantown|
|  San Jose City Hall|
|         MLK Library|
|SJSU 4th at San C...|
|       St James Park|
|Arena Green / SAP...|
|SJSU - San Salvad...|
|   Franklin at Maple|
|Redwood City Calt...|
|San Mateo County ...|
|Redwood City Publ...|
|Stanford in Redwo...|
|Redwood City Medi...|
+--------------------+
only showing top 20 rows



8) Return the station with the maximum number of incoming trips.

In [ ]:
# "Incoming trips" => destination

incoming_trips = graph.edges.groupBy("dst").count().orderBy("count", ascending=False).limit(1)

print("Station with the maximum number of incoming trips:")
incoming_trips.show()

Station with the maximum number of incoming trips:
+--------------------+-----+
|                 dst|count|
+--------------------+-----+
|San Francisco Cal...|    9|
+--------------------+-----+



9) Return the trip with the longest duration.

In [ ]:
longest_trip = graph.edges.orderBy("Duration", ascending=False).limit(1)

print("Trip with the longest duration:")
longest_trip.show()

Trip with the longest duration:
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
|Trip ID|Duration|     Start Date|                 src|Start Terminal|       End Date|                 dst|End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
| 913386|    1808|8/31/2015 20:23|Embarcadero at Br...|            54|8/31/2015 20:53|Harry Bridges Pla...|          50|   524|     Subscriber|   94105|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+



10)Create a subgraph that contains only the trips that start or end at "Townsend at 7th".

In [ ]:
sub_edges = graph.edges.filter("src = 'Townsend at 7th' OR dst = 'Townsend at 7th'") #trips

sub_vertices_ids = sub_edges.select("src").union(sub_edges.select("dst")).distinct() #unique stations
sub_vertices = graph.vertices.join(sub_vertices_ids, graph.vertices.id == sub_vertices_ids.src, "inner") #join (to get full details)

subgraph = GraphFrame(sub_vertices, sub_edges)

print("Subgraph Edges (trips starting or ending at 'Townsend at 7th'):")
subgraph.edges.show()
print("Subgraph Vertices (stations involved):")
subgraph.vertices.show()

Subgraph Edges (trips starting or ending at 'Townsend at 7th'):
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
|Trip ID|Duration|     Start Date|                 src|Start Terminal|       End Date|                 dst|End Terminal|Bike #|Subscriber Type|Zip Code|
+-------+--------+---------------+--------------------+--------------+---------------+--------------------+------------+------+---------------+--------+
| 913434|     283|8/31/2015 21:19|San Francisco Cal...|            69|8/31/2015 21:24|     Townsend at 7th|          65|   521|     Subscriber|   94107|
| 913404|     273|8/31/2015 20:39|San Francisco Cal...|            69|8/31/2015 20:44|     Townsend at 7th|          65|   287|     Subscriber|   94107|
| 913382|     645|8/31/2015 20:20|     Townsend at 7th|            65|8/31/2015 20:31|Harry Bridges Pla...|          50|   270|     Subscriber|   94945|
| 913379|     803|

11) Return all paths that form a "triangle" pattern between three stations.

In [ ]:
triangles = graph.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")
#(a)=> vertices (station)
#[e1]=>Edges (Trips)
print("All triangle patterns between three stations:")
triangles.show(truncate=False)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


All triangle patterns between three stations:
+------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------+
|a                                                                                               |e1                                                                                                                                       |b        

12) Return all paths that pass through three vertices and start from "Townsend at 7th".

In [ ]:
paths = graph.find("(a)-[e1]->(b); (b)-[e2]->(c)") \
    .filter("a.id = 'Townsend at 7th'")

print("Paths passing through three vertices starting from 'Townsend at 7th':")
paths.show(truncate=False)

Paths passing through three vertices starting from 'Townsend at 7th':
+---------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------+
|a                                                                          |e1                                                                                                                                    |b                                                                                               |e2                                                  